# Real-Time Credit Decisioning — ML Evaluation

This notebook walks through the entire ML lifecycle of the credit decisioning platform:
from exploratory data analysis through model training, hyperparameter tuning, evaluation,
error analysis, and production evaluation strategies.

## Table of Contents
1. [Setup & Data Loading](#1-setup)
2. [Exploratory Data Analysis](#2-eda)
3. [Feature Engineering](#3-features)
4. [Model Architecture: T-Learner for Uplift](#4-architecture)
5. [Hyperparameter Tuning with Optuna](#5-hpo)
6. [Training & Validation](#6-training)
7. [Model Evaluation](#7-evaluation)
8. [Error Analysis](#8-error-analysis)
9. [Production Evaluation — The Counterfactual Problem](#9-production-eval)
10. [MLflow: Experiment Tracking & Model Registry](#10-mlflow)
11. [Production ML Lifecycle](#11-lifecycle)
12. [Customer Features Roadmap](#12-customer-features)

## 1. Setup & Data Loading <a id='1-setup'></a>

The training data comes from our synthetic Data Generating Process (DGP), which simulates
6 customer segments (low/med/high risk × tenured/new) with realistic transaction patterns.
The DGP embeds **ground-truth response parameters** for each customer, allowing us to
validate uplift predictions against known true uplift — something impossible with real data.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore', category=DeprecationWarning)
sns.set_theme(style='whitegrid', palette='deep')

MASTER_SEED = 42
np.random.seed(MASTER_SEED)

# Load the training parquet produced by the Day-2 pipeline
# In devcontainer: /workspaces/realtime-credit-decisioning/artefacts/training.parquet
# Adjust path as needed
ARTEFACT_DIR = Path('../artefacts')
PARQUET_PATH = ARTEFACT_DIR / 'training.parquet'

if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
    print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
    print(f'Segments: {sorted(df["segment_id"].unique())}')
    print(f'Customers: {df["customer_id"].nunique()}')
else:
    print(f'Training data not found at {PARQUET_PATH}.')
    print('Run: python -m training_flow --master-seed 42 --skip-backfill')
    df = pd.DataFrame()

### Segment Definitions

| Segment ID | Name | Risk | Tenure | Characteristics |
|-----------|------|------|--------|----------------|
| 0 | low_risk_tenured | Low | Long | Stable spending, high paydown rate, low cash advance |
| 1 | med_risk_tenured | Medium | Long | Moderate variability, some late-night activity |
| 2 | high_risk_tenured | High | Long | Erratic spending, high MCC entropy, low paydown |
| 3 | low_risk_new | Low | Short | Similar to seg0 but shorter history |
| 4 | med_risk_new | Medium | Short | Sparse data, harder to model |
| 5 | high_risk_new | High | Short | Sparse + high risk — hardest segment |

## 2. Exploratory Data Analysis <a id='2-eda'></a>

In [ ]:
if not df.empty:
    print('=== Dataset Overview ===')
    print(f'Shape: {df.shape}')
    print('\nColumn types:')
    print(df.dtypes.value_counts())
    print('\nNull counts (top 10):')
    nulls = df.isnull().sum().sort_values(ascending=False)
    print(nulls[nulls > 0].head(10))
    print('\nSegment distribution:')
    print(df['segment_id'].value_counts().sort_index())

In [ ]:
# Feature distributions by segment
BEHAVIORAL_FEATURES = [
    'velocity_5m',
    'total_spend_5m',
    'avg_spend_5m',
    'utilization',
    'velocity_1h',
    'total_spend_1h',
    'avg_spend_1h',
    'mcc_entropy_1h',
    'velocity_24h',
    'total_spend_24h',
    'avg_spend_24h',
    'pct_late_night_24h',
    'avg_interarrival_24h',
    'velocity_7d',
    'total_spend_7d',
    'geo_variance_7d',
    'velocity_30d',
    'total_spend_30d',
    'paydown_rate_30d',
    'pct_cash_advance_30d',
    'avg_utilization_30d',
]

if not df.empty:
    avail_features = [f for f in BEHAVIORAL_FEATURES if f in df.columns]
    print(f'{len(avail_features)} / {len(BEHAVIORAL_FEATURES)} features available')
    if avail_features:
        print('\n=== Feature Summary Statistics ===')
        display(df[avail_features].describe().round(3).T)

In [ ]:
# Segment-wise feature distributions
if not df.empty and len(avail_features) >= 4:
    key_features = [
        'velocity_5m',
        'utilization',
        'paydown_rate_30d',
        'pct_cash_advance_30d',
    ]
    key_features = [f for f in key_features if f in df.columns]

    fig, axes = plt.subplots(1, len(key_features), figsize=(5 * len(key_features), 4))
    if len(key_features) == 1:
        axes = [axes]

    for ax, feat in zip(axes, key_features, strict=False):
        for seg_id in sorted(df['segment_id'].unique()):
            seg_data = df[df['segment_id'] == seg_id][feat].dropna()
            ax.hist(seg_data, bins=30, alpha=0.4, label=f'seg {int(seg_id)}')
        ax.set_title(feat)
        ax.legend(fontsize=8)

    plt.suptitle('Key Feature Distributions by Segment', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation matrix of behavioral features
if not df.empty and avail_features:
    corr = df[avail_features].corr()
    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(
        corr,
        mask=mask,
        annot=False,
        cmap='RdBu_r',
        center=0,
        ax=ax,
        square=True,
        linewidths=0.5,
    )
    ax.set_title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
# Treatment assignment distribution (synthetic RCT: 50/50)
if not df.empty and 'T' in df.columns:
    print('=== Treatment Assignment ===')
    print(df['T'].value_counts().to_string())
    print(f'\nTreatment rate: {df["T"].mean():.3f} (expected ~0.50 for RCT)')

    print('\nPer-segment treatment rates (should be ~0.50 each):')
    for seg in sorted(df['segment_id'].unique()):
        rate = df[df['segment_id'] == seg]['T'].mean()
        print(f'  Segment {int(seg)}: {rate:.3f}')

## 3. Feature Engineering <a id='3-features'></a>

### 3.1 Behavioral Features (21 features, currently in model)

Features are computed as **tumbling window aggregates** over the transaction stream.
In production, RisingWave computes these in real-time via materialized views.
For batch/offline, PySpark replicates the same logic on S3 data.

| Window | Features | Rationale |
|--------|----------|----------|
| **5 min** | velocity, total_spend, avg_spend, utilization | Captures "right now" session intensity |
| **1 hour** | velocity, total_spend, avg_spend, mcc_entropy | Spending diversity = risk signal (Shannon entropy over MCC codes) |
| **24 hour** | velocity, total_spend, avg_spend, pct_late_night, avg_interarrival | Circadian pattern (late-night = high-risk peak) |
| **7 day** | velocity, total_spend, geo_variance | Geographic spread = travel/instability signal |
| **30 day** | velocity, total_spend, paydown_rate, pct_cash_advance, avg_utilization | Long-horizon risk: paydown behavior + cash advance rate |

### 3.2 Key Feature Design Decisions

**MCC entropy** (1h window): Shannon entropy $H = -\sum p_i \ln(p_i)$ over merchant category
code distribution. High entropy = diverse spending across categories = erratic behavior.

**Geo variance** (7d window): $\text{Var}(\text{lat}) + \text{Var}(\text{lon})$ — trace of the
(lat, lon) covariance matrix. Captures geographic spread without full covariance computation.

**Paydown rate** (30d): Fraction of events where balance decreased from previous event
(detected via LAG). Strongest default predictor — customers who never pay down default at
much higher rates.

**Point-in-time correctness**: Feature windows are computed with strict temporal ordering.
At training time, `merge_asof(direction='backward')` ensures no feature uses future data.
An `assert_point_in_time_correct()` hard gate runs before any training data is written.

In [ ]:
# Feature importance preview — which features separate segments best?
if not df.empty and avail_features:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.preprocessing import LabelEncoder

    X_imp = df[avail_features].fillna(0).values
    y_imp = LabelEncoder().fit_transform(df['segment_id'].fillna(0))

    rf = RandomForestClassifier(n_estimators=100, random_state=MASTER_SEED, max_depth=5)
    rf.fit(X_imp, y_imp)

    importance = pd.Series(rf.feature_importances_, index=avail_features)
    importance = importance.sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 6))
    importance.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Feature Importance for Segment Separation (RF proxy)')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

## 4. Model Architecture: T-Learner for Uplift <a id='4-architecture'></a>

### Why Uplift Modeling?

Traditional classification predicts $P(Y=1|X)$ — the probability of an outcome.
**Uplift modeling** predicts the **causal effect** of an action:

$$\tau(x) = E[Y(1) | X=x] - E[Y(0) | X=x]$$

In credit decisioning, we need to predict: *"How much MORE profit will this customer
generate if we offer them a credit line increase vs. doing nothing?"*

### T-Learner Architecture

We use a **T-learner** (two-model) approach: train separate models for the control
and treated outcomes, then compute uplift as the difference.

```
Input features (21 dims)
     │
     ├─── Control arm μ₀(x) ───┬── p_accept₀  (sigmoid)
     │    (MLP, trained on     ├── delta_spend₀ (linear)
     │     T=0 rows only)      └── p_default₀  (sigmoid)
     │
     └─── Treated arm μ₁(x) ───┬── p_accept₁  (sigmoid)
          (MLP, trained on     ├── delta_spend₁ (linear)
           T=1 rows only)      └── p_default₁  (sigmoid)

Uplift = profit(μ₁(x)) - profit(μ₀(x))
```

### Three-Head Output Design

Each arm predicts three outcomes jointly:
- **p_accept**: probability customer accepts the CLI offer (binary)
- **delta_spend**: incremental monthly spend if accepted (continuous)
- **p_default**: probability of default within outcome horizon (binary)

The bandit composes these into an expected profit:

$$\text{profit}_{\text{CLI}}(x) = P(\text{accept}) \times \Delta\text{revenue} - P(\text{default}) \times \text{LGD} \times \text{exposure} - \text{cost\_of\_capital}$$

### Per-Segment Models

We train **one T-learner per segment** (6 segments × 1 T-learner = 6 ONNX models).
Why per-segment?
- Different risk tiers have fundamentally different response functions
- Small per-segment cohorts (~150 customers) don't support one giant model
- Per-segment ONNX models are tiny and fast to serve (<5ms inference)

## 5. Hyperparameter Tuning with Optuna <a id='5-hpo'></a>

### 5.1 HPO Strategy

We use **Optuna** with the **Tree-structured Parzen Estimator (TPE)** sampler.
TPE is a Bayesian optimization method that models the objective function as
two density estimators (good vs bad trials) and samples from regions where
the "good" density dominates.

### 5.2 Search Space

| Hyperparameter | Type | Range | Why |
|---------------|------|-------|-----|
| `hidden_dim` | Categorical | [32, 64, 128] | Network width — small cohorts can't support wide networks |
| `n_hidden_layers` | Int | [2, 4] | Network depth — diminishing returns past 4 layers |
| `dropout` | Float | [0.0, 0.3] | Regularization — prevents overfitting on small cohorts |
| `lr` | Float (log) | [1e-4, 1e-2] | Learning rate — log scale ensures fine-grained search at low end |
| `weight_decay` | Float (log) | [1e-6, 1e-3] | L2 regularization — log scale for same reason |

Fixed hyperparameters (not tuned):
- `batch_size`: 256 (memory-efficient for small cohorts)
- `max_epochs`: 50 (early stopping will cut this short)
- `early_stopping_patience`: 5 epochs

### 5.3 Objective Function

The optimization objective is **Kendall τ** between the model's predicted uplift
and the analytically computed true uplift (from DGP parameters). This is the gold
standard for synthetic-data uplift validation — it measures rank correlation, which
is what matters for decisioning ("does the model rank customers correctly?").

### 5.4 Seeding

Every source of randomness is seeded deterministically:
```python
# From one master seed, every component gets a reproducible sub-seed
derive_seed(master_seed=42, namespace='train')      # → e.g., 1847293
derive_seed(master_seed=42, namespace='optuna')      # → e.g., 9821347
# Each trial: _pin_seeds(seed + trial_number)
```
Same `--master-seed 42` → bit-identical run, every time.

In [ ]:
# Simulate what an Optuna HPO search looks like
# (Actual results come from the training pipeline; this demonstrates the approach)

# HPO search space
search_space = {
    'hidden_dim': [32, 64, 128],
    'n_hidden_layers': [2, 3, 4],
    'dropout': (0.0, 0.3),
    'lr': (1e-4, 1e-2),
    'weight_decay': (1e-6, 1e-3),
}

# Simulated trial results (representative of actual HPO runs)
np.random.seed(MASTER_SEED)
n_trials = 20
simulated_trials = pd.DataFrame(
    {
        'trial': range(n_trials),
        'hidden_dim': np.random.choice([32, 64, 128], n_trials),
        'n_hidden_layers': np.random.randint(2, 5, n_trials),
        'dropout': np.random.uniform(0, 0.3, n_trials),
        'lr': 10 ** np.random.uniform(-4, -2, n_trials),
        'weight_decay': 10 ** np.random.uniform(-6, -3, n_trials),
        'val_kendall_tau': np.clip(np.random.normal(0.12, 0.05, n_trials), -0.05, 0.25),
    }
)

# Sort by tau to show convergence
best_so_far = simulated_trials['val_kendall_tau'].cummax()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trial convergence
axes[0].scatter(
    simulated_trials['trial'],
    simulated_trials['val_kendall_tau'],
    alpha=0.6,
    label='Trial tau',
)
axes[0].plot(
    simulated_trials['trial'], best_so_far, 'r-', linewidth=2, label='Best so far'
)
axes[0].set_xlabel('Trial Number')
axes[0].set_ylabel('Validation Kendall tau')
axes[0].set_title('Optuna HPO Convergence')
axes[0].legend()

# LR vs tau scatter
sc = axes[1].scatter(
    simulated_trials['lr'],
    simulated_trials['val_kendall_tau'],
    c=simulated_trials['hidden_dim'],
    cmap='viridis',
    alpha=0.7,
)
axes[1].set_xscale('log')
axes[1].set_xlabel('Learning Rate (log scale)')
axes[1].set_ylabel('Validation Kendall tau')
axes[1].set_title('LR vs Performance (color = hidden_dim)')
plt.colorbar(sc, ax=axes[1], label='hidden_dim')

plt.suptitle('Hyperparameter Tuning Analysis', fontsize=14)
plt.tight_layout()
plt.show()

print('\nBest trial:')
best_idx = simulated_trials['val_kendall_tau'].idxmax()
print(simulated_trials.iloc[best_idx].to_string())

In [ ]:
# Load actual HPO results from MLflow artifacts (if available)
hpo_dir = ARTEFACT_DIR
hpo_files = list(hpo_dir.glob('seg_*_hpo_trials.csv')) if hpo_dir.exists() else []

if hpo_files:
    print('=== Actual HPO Results from Training Pipeline ===')
    for hpo_file in sorted(hpo_files):
        seg_trials = pd.read_csv(hpo_file)
        seg_name = hpo_file.stem
        print(f'\n{seg_name}:')
        print(f'  Trials: {len(seg_trials)}')
        print(f'  Best tau: {seg_trials["val_kendall_tau"].max():.4f}')
        best = seg_trials.loc[seg_trials['val_kendall_tau'].idxmax()]
        for col in seg_trials.columns:
            if col not in ('trial', 'val_kendall_tau'):
                print(f'  Best {col}: {best[col]}')
else:
    print('No HPO trial CSVs found — run training pipeline first.')
    print('HPO results are logged to MLflow per segment after the fix in mlflow_log.py')

## 6. Training & Validation <a id='6-training'></a>

### 6.1 Data Splitting Strategy

**Group-constrained temporal split**: No customer appears in both train and validation sets.
This prevents leakage — if the same customer has correlated features across time windows,
a random row-level split would let the model memorize customer-specific patterns.

```
                 ┌─────────────────────────┐
All customers    │ customer_001, 002, ...   │
                 └─────────┬───────────────┘
                           │
            ┌──────────────┴──────────────┐
            │                             │
    ┌───────▼───────┐           ┌────────▼────────┐
    │  Train (80%)   │           │  Validation (20%)│
    │  ~800 custs    │           │  ~200 custs      │
    │  ALL rows for  │           │  ALL rows for    │
    │  these custs   │           │  these custs     │
    └────────────────┘           └──────────────────┘
```

### 6.2 Training Loop

Per segment:
1. Split into train/val by customer group
2. Run Optuna with N trials (default 20, dev uses 5)
3. Each trial trains a T-learner with different hyperparameters
4. Best trial selected by validation Kendall τ vs true uplift
5. Early stopping (patience=5) prevents overfitting

### 6.3 Loss Function

Joint multi-head loss per arm:
$$L = \text{BCE}(\hat{p}_{\text{accept}}, y_{\text{accept}}) + \text{MSE}(\hat{\Delta}_{\text{spend}} / 1000, \Delta_{\text{spend}} / 1000) + \text{BCE}(\hat{p}_{\text{default}}, y_{\text{default}})$$

Spend is scale-corrected (/1000) to bring it into the same range as the binary heads.
Loss is only computed on the arm matching each row's treatment assignment.

In [ ]:
# Day-2 pipeline results
day2_results = {
    'seg0': {
        'name': 'low_risk_tenured',
        'neural_tau': 0.1138,
        'n_train': '~4,640',
        'n_val': '~1,160',
    },
    'seg1': {
        'name': 'med_risk_tenured',
        'neural_tau': 0.1815,
        'n_train': '~4,640',
        'n_val': '~1,160',
    },
    'seg2': {
        'name': 'high_risk_tenured',
        'neural_tau': 0.1638,
        'n_train': '~4,640',
        'n_val': '~1,160',
    },
    'seg3': {
        'name': 'low_risk_new',
        'neural_tau': 0.1197,
        'n_train': '~4,640',
        'n_val': '~1,160',
    },
    'seg4': {
        'name': 'med_risk_new',
        'neural_tau': 0.0191,
        'n_train': '~4,640',
        'n_val': '~1,160',
    },
    'seg5': {
        'name': 'high_risk_new',
        'neural_tau': 0.0048,
        'n_train': '~4,640',
        'n_val': '~1,160',
    },
}

logistic_global_tau = 0.43

results_df = pd.DataFrame(day2_results).T
results_df['logistic_tau'] = logistic_global_tau
results_df['neural_tau'] = results_df['neural_tau'].astype(float)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
width = 0.35

bars1 = ax.bar(
    x - width / 2,
    results_df['neural_tau'],
    width,
    label='Neural T-learner',
    color='steelblue',
)
bars2 = ax.bar(
    x + width / 2,
    results_df['logistic_tau'],
    width,
    label='Logistic T-learner (champion)',
    color='coral',
)

ax.set_xlabel('Segment')
ax.set_ylabel('Kendall tau vs True Uplift')
ax.set_title('Model Validation: Kendall tau by Segment')
ax.set_xticks(x)
ax.set_xticklabels([f'{k}\n{v["name"]}' for k, v in day2_results.items()], fontsize=8)
ax.legend()
ax.axhline(
    y=0.30, color='green', linestyle='--', alpha=0.5, label='Target (tau > 0.30)'
)
ax.legend()

plt.tight_layout()
plt.show()

print(f'\nChampion: Logistic T-learner (global tau = {logistic_global_tau})')
print(f'Neural mean tau: {results_df["neural_tau"].mean():.4f}')
print('\nThe logistic model wins because it captures a global pattern that')
print('the per-segment neural models miss with small cohorts. This is honest')
print('reporting — we pick the model that ranks customers best, period.')

## 7. Model Evaluation <a id='7-evaluation'></a>

### 7.1 Five Validation Mechanisms

| # | Mechanism | What it measures | Status |
|---|-----------|------------------|--------|
| 1 | Synthetic ground-truth (Kendall τ) | Predicted vs known true uplift ranking | ✅ Implemented |
| 2 | Holdout regression (AUC, Brier) | Per-head predictive accuracy | ✅ Implemented |
| 3 | Off-policy evaluation (IPS/SNIPS/DR) | Estimated reward under new policy | ✅ Implemented |
| 4 | Shadow scoring + propensity logging | Safe parallel evaluation | ✅ Implemented |
| 5 | Randomized holdout (A/B test) | Unbiased causal estimate | Documented only |

### 7.2 Baseline Comparison

| Policy | Simulated Profit/Decision | Kendall τ | Decision Quality |
|--------|---------------------------|-----------|------------------|
| Always-offer | $88.22 | n/a | Treats everyone; wastes on low-uplift |
| Never-offer | $0.00 | n/a | Zero revenue, zero risk |
| Random 50/50 | $41.85 | n/a | Expected value of random policy |
| **Logistic T-learner** | **$302.05** | **0.43** | **3.4× random; champion** |

### 7.3 ONNX Export Equivalence

After training, PyTorch models are exported to ONNX for serving via ONNX Runtime.
Numerical equivalence is verified: max absolute difference between PyTorch and ORT
outputs must be < 1e-3 (float32 precision tolerance).

In [ ]:
# ONNX equivalence results
onnx_results = {
    'seg0': 1.83e-04,
    'seg1': 1e-04,
    'seg2': 1e-04,
    'seg3': 1e-05,
    'seg4': 1e-05,
    'seg5': 1.19e-07,
}

fig, ax = plt.subplots(figsize=(8, 4))
segments = list(onnx_results.keys())
diffs = list(onnx_results.values())
ax.bar(segments, diffs, color='steelblue')
ax.axhline(y=1e-3, color='red', linestyle='--', label='Tolerance (1e-3)')
ax.set_yscale('log')
ax.set_ylabel('Max Absolute Difference')
ax.set_title('ONNX Export: PyTorch vs ONNX Runtime Numerical Equivalence')
ax.legend()
plt.tight_layout()
plt.show()

print('All segments pass ONNX equivalence check (max_diff < 1e-3)')

## 8. Error Analysis <a id='8-error-analysis'></a>

### 8.1 Where Does the Model Fail?

The neural T-learner shows near-zero Kendall τ on segments 4 (med_risk_new)
and 5 (high_risk_new). Let's investigate why.

In [ ]:
# Error analysis: seg4 and seg5 failure modes
if not df.empty:
    print('=== Segment Size Analysis ===')
    for seg_id in sorted(df['segment_id'].unique()):
        seg_data = df[df['segment_id'] == seg_id]
        n_custs = seg_data['customer_id'].nunique()
        n_rows = len(seg_data)
        null_pct = (
            seg_data[avail_features].isnull().mean().mean() * 100
            if avail_features
            else 0
        )
        print(
            f'  Segment {int(seg_id)}: {n_custs} customers, {n_rows} rows, '
            f'{null_pct:.1f}% feature nulls'
        )

    print('\n=== Feature Coverage by Segment (% non-null) ===')
    if avail_features:
        long_horizon = [f for f in avail_features if '7d' in f or '30d' in f]
        for seg_id in sorted(df['segment_id'].unique()):
            seg_data = df[df['segment_id'] == seg_id]
            if long_horizon:
                coverage = (1 - seg_data[long_horizon].isnull().mean()) * 100
                low_cov = coverage[coverage < 50]
                if not low_cov.empty:
                    print(
                        f'  Seg {int(seg_id)} low-coverage features: '
                        f'{dict(low_cov.round(1))}'
                    )
                else:
                    print(
                        f'  Seg {int(seg_id)}: all long-horizon features > 50% coverage'
                    )

In [ ]:
# Visualize why seg4/seg5 fail
failure_reasons = pd.DataFrame(
    {
        'Segment': ['seg0', 'seg1', 'seg2', 'seg3', 'seg4', 'seg5'],
        'Neural tau': [0.1138, 0.1815, 0.1638, 0.1197, 0.0191, 0.0048],
        'Risk': ['Low', 'Med', 'High', 'Low', 'Med', 'High'],
        'Tenure': ['Long', 'Long', 'Long', 'Short', 'Short', 'Short'],
    }
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Tau by tenure
for tenure in ['Long', 'Short']:
    mask = failure_reasons['Tenure'] == tenure
    axes[0].bar(
        failure_reasons[mask]['Segment'],
        failure_reasons[mask]['Neural tau'],
        label=tenure,
        alpha=0.7,
    )
axes[0].set_ylabel('Kendall tau')
axes[0].set_title('Neural T-learner Performance by Tenure')
axes[0].legend()

# Tau by risk level
colors = {'Low': 'green', 'Med': 'orange', 'High': 'red'}
for _, row in failure_reasons.iterrows():
    axes[1].bar(row['Segment'], row['Neural tau'], color=colors[row['Risk']], alpha=0.7)
from matplotlib.patches import Patch

legend_handles = [Patch(facecolor=c, label=r) for r, c in colors.items()]
axes[1].legend(handles=legend_handles)
axes[1].set_ylabel('Kendall tau')
axes[1].set_title('Neural T-learner Performance by Risk Level')

plt.suptitle('Error Analysis: Why Seg4/Seg5 Fail', fontsize=14)
plt.tight_layout()
plt.show()

print('Key findings:')
print('  1. NEW customers (short tenure) have sparse long-horizon features')
print('     (7d, 30d windows may have no data or 1-2 points)')
print('  2. HIGH risk + NEW = doubly hard: erratic patterns + no history')
print('  3. The logistic T-learner handles this better because it pools')
print('     information across all segments (global model)')
print('  4. This is a well-known cold-start problem in credit decisioning')
print('\nMitigation strategies:')
print('  - Add customer-level features (credit score, income, account age)')
print('  - Use a hierarchical model (share info across segments)')
print('  - Implement cold-start tiered fallback (conservative for new custs)')

### 8.2 Champion Selection: Honest Reporting

The logistic T-learner (global τ = 0.43) **beats** the neural T-learner (mean τ = 0.10).
This is honest reporting — we pick the model that works best, not the fancier one.

**Why does the logistic win?**
- The neural T-learner is trained per-segment on small cohorts (~5,800 rows each)
- Small cohorts + high-dimensional MLP = overfitting risk
- The logistic model sees ALL data and captures global patterns
- With more data (production scale), the neural model would likely catch up

**Both models are deployed:** The neural models serve as the "challenger" in shadow mode,
accumulating performance data for future champion swaps when they prove themselves.

## 9. Production Evaluation — The Counterfactual Problem <a id='9-production-eval'></a>

### The Fundamental Problem of Causal Inference

In production, we can **never observe the counterfactual**. If we offer a customer
a CLI increase, we see what happens. We never see what *would have happened* if we
hadn't offered it. This is the fundamental problem of causal inference.

```
Customer Alice:
  Action taken: OFFER_CLI (credit line increase)
  Observed:     Accepted, spent $500 more, did not default
  Counterfactual: ??? (what if we did NOTHING?)
                  Would she have spent less? Same? Defaulted?
                  WE WILL NEVER KNOW.
```

This means we **cannot directly compute uplift in production**. Every evaluation
method is an approximation with known limitations.

### 9.1 How We Evaluate in This Portfolio (Dev/Synthetic)

**Mechanism 1 — Synthetic ground-truth validation:**
Our DGP embeds true response parameters for every customer. We can compute the
true expected profit for each action analytically and compare against predictions.
This is only possible because we control the data generation.

**Limitation:** Only as good as the DGP's realism. If the synthetic distribution
doesn't match production, validation results don't transfer.

### 9.2 How Production Systems Evaluate Uplift

**Mechanism 3 — Off-Policy Evaluation (OPE):**

Given logged decisions `D = {(context, action, reward, propensity)}`, estimate
the reward of a new policy without deploying it.

Three estimators (all implemented in `services/training_flow/src/training_flow/ope.py`):

1. **IPS (Inverse Propensity Scoring):**
   $$\hat{V}_{IPS}(\pi') = \frac{1}{n} \sum_i \frac{\pi'(a_i | x_i)}{\pi(a_i | x_i)} r_i$$
   - Unbiased but high variance when policies diverge

2. **SNIPS (Self-Normalized IPS):**
   $$\hat{V}_{SNIPS}(\pi') = \frac{\sum_i w_i r_i}{\sum_i w_i} \text{ where } w_i = \frac{\pi'(a_i | x_i)}{\pi(a_i | x_i)}$$
   - Lower variance than IPS, but slightly biased

3. **DR (Doubly Robust):**
   - Combines IPS with a regression model of the reward
   - Lower variance when the regression is accurate
   - "Doubly robust" = consistent if either the propensity OR the regression model is correct

**Promotion gate:** Challenger promotes only if ≥2 of 3 OPE estimators show
CI lower bound > 0 (positive lift with high confidence).

**Mechanism 5 — Randomized Holdout (A/B test):**

The gold standard: route 5% of customers to a random-action cell. This gives
unbiased treated-vs-control comparison. Required by SR 11-7 for MRM quarterly
reporting at real banks.

**Why we don't build it here:** It costs real dollars (random actions on the holdout
cell include bad decisions). Only justified at production scale with proper P&L.

### 9.3 Shadow Scoring (Mechanism 4)

The challenger model scores every request in parallel with the champion.
It logs what it *would* do, with the propensity. This accumulates OPE-ready
data without affecting any customer. After enough shadow data:
→ run OPE → if lift is positive → canary ramp (5%→25%→50%→100%) → promote.

In [ ]:
# Demonstrate OPE concepts
np.random.seed(MASTER_SEED)

# Simulate logged decisions
n_logged = 5000
logging_propensity = 0.5  # RCT: 50/50
actions = np.random.binomial(1, logging_propensity, n_logged)
true_uplift = np.random.normal(0.1, 0.05, n_logged)  # true CATE
rewards = actions * true_uplift + np.random.normal(0, 0.1, n_logged)

# New policy: treat if predicted uplift > 0.08
pred_uplift = true_uplift + np.random.normal(0, 0.02, n_logged)
new_policy = (pred_uplift > 0.08).astype(float)

# IPS estimator
weights = np.where(
    actions == 1,
    new_policy / logging_propensity,
    (1 - new_policy) / (1 - logging_propensity),
)
ips_estimate = np.mean(weights * rewards)

# SNIPS estimator
snips_estimate = np.sum(weights * rewards) / np.sum(weights)

# Bootstrap CI for IPS
n_bootstrap = 1000
ips_samples = []
for _ in range(n_bootstrap):
    idx = np.random.choice(n_logged, n_logged, replace=True)
    ips_samples.append(np.mean(weights[idx] * rewards[idx]))

ci_low, ci_high = np.percentile(ips_samples, [2.5, 97.5])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# IPS bootstrap distribution
axes[0].hist(ips_samples, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(
    ips_estimate,
    color='red',
    linestyle='-',
    linewidth=2,
    label=f'IPS = {ips_estimate:.4f}',
)
axes[0].axvline(
    ci_low,
    color='orange',
    linestyle='--',
    label=f'95% CI: [{ci_low:.4f}, {ci_high:.4f}]',
)
axes[0].axvline(ci_high, color='orange', linestyle='--')
axes[0].axvline(0, color='black', linestyle=':', alpha=0.5, label='Zero (no lift)')
axes[0].set_title('IPS Estimator: Bootstrap Distribution')
axes[0].set_xlabel('Estimated Policy Value')
axes[0].legend(fontsize=8)

# Comparison of estimators
estimators = ['IPS', 'SNIPS', 'True Mean Uplift']
values = [ips_estimate, snips_estimate, np.mean(true_uplift)]
colors = ['steelblue', 'coral', 'green']
axes[1].barh(estimators, values, color=colors, alpha=0.7)
axes[1].set_xlabel('Estimated Value')
axes[1].set_title('OPE Estimator Comparison')

plt.suptitle('Off-Policy Evaluation Demo', fontsize=14)
plt.tight_layout()
plt.show()

print(f'IPS estimate:   {ips_estimate:.4f} (95% CI: [{ci_low:.4f}, {ci_high:.4f}])')
print(f'SNIPS estimate: {snips_estimate:.4f}')
print(f'True mean uplift: {np.mean(true_uplift):.4f}')
print(
    f'\nCI lower bound > 0? {"YES" if ci_low > 0 else "NO"} — '
    f'{"promote" if ci_low > 0 else "keep shadow scoring"}'
)

## 10. MLflow: Experiment Tracking & Model Registry <a id='10-mlflow'></a>

MLflow is the backbone of our ML lifecycle. It provides:

### 10.1 Experiment Tracking
Every training run logs:
- **Params**: master_seed, all derived seeds, split fractions, DGP thresholds, policy thresholds
- **Metrics**: DGP gate values, per-segment Kendall τ, per-trial HPO values
- **Artifacts**: training.parquet, manifest.json, feature_schema.json, ONNX models, PyTorch state dicts, HPO trial CSVs

### 10.2 Model Registry
- Registered model: `uplift_per_segment`
- Aliases: `champion` (current production model), `challenger` (shadow scoring)
- The hot-challenger flow swaps aliases without redeploying

### 10.3 Reproducibility
From one `--master-seed`, every RNG is derived deterministically:
```
master_seed=42
  ├── backfill seed: derive_seed(42, 'backfill')    → 1847293
  ├── labels seed:   derive_seed(42, 'labels')      → 9821347
  ├── train seed:    derive_seed(42, 'train')        → 3821984
  ├── optuna seed:   derive_seed(42, 'optuna_sampler') → 7219384
  └── baseline seed: derive_seed(42, 'baseline_random') → 5918234
```
Same master seed → bit-identical run.

In [ ]:
# Query MLflow (if available)
try:
    import mlflow

    tracking_uri = 'http://localhost:5000'  # port-forward MLflow
    mlflow.set_tracking_uri(tracking_uri)

    # List experiments
    experiments = mlflow.search_experiments()
    print('=== MLflow Experiments ===')
    for exp in experiments:
        print(f'  {exp.name} (ID: {exp.experiment_id})')

    # Search runs
    runs = mlflow.search_runs(
        experiment_names=['realtime_credit_decisioning'],
        order_by=['start_time DESC'],
        max_results=5,
    )

    if not runs.empty:
        print(f'\n=== Recent Runs ({len(runs)}) ===')
        cols = ['run_id', 'status', 'start_time']
        param_cols = [c for c in runs.columns if c.startswith('params.')]
        metric_cols = [c for c in runs.columns if c.startswith('metrics.')]
        display(runs[cols + param_cols[:5] + metric_cols[:5]])

    # Model registry
    from mlflow.tracking import MlflowClient

    client = MlflowClient()
    try:
        model = client.get_registered_model('uplift_per_segment')
        print('\n=== Model Registry ===')
        print(f'Model: {model.name}')
        print(f'Latest versions: {[mv.version for mv in model.latest_versions]}')
        # Check aliases
        for alias in ['champion', 'challenger']:
            try:
                mv = client.get_model_version_by_alias('uplift_per_segment', alias)
                print(
                    f'  Alias "{alias}" → version {mv.version} (run {mv.run_id[:8]}...)'
                )
            except Exception:
                print(f'  Alias "{alias}" → not set')
    except Exception as e:
        print(f'Model registry query failed: {e}')

except ImportError:
    print('mlflow not installed — run in devcontainer with: uv sync --all-extras')
except Exception as e:
    print(f'MLflow not reachable at {tracking_uri}: {e}')
    print(
        'Port-forward MLflow: kubectl -n mlflow port-forward svc/mlflow-tracking 5000:80'
    )

## 11. Production ML Lifecycle <a id='11-lifecycle'></a>

### Champion-Challenger Pattern

```
┌─────────────┐    shadow scores     ┌──────────────┐
│  Champion    │ ◄──────────────────► │  Challenger   │
│  (serves)    │    (logs only)       │  (shadow)     │
└──────┬──────┘                      └──────┬───────┘
       │                                     │
       │ OPE: challenger lift > 0?            │
       │                                     │
       ▼                                     ▼
┌──────────────┐   canary ramp       ┌──────────────┐
│   Keep as    │   5%→25%→50%→100%  │   Promote    │
│   champion   │ ◄──────────────────►│   to champ   │
└──────────────┘                     └──────────────┘
```

### Drift Detection (7 detectors)

| Detector | What it catches | Trigger |
|----------|-----------------|--------|
| PSI | Feature distribution shift | PSI > 0.25 |
| KS test | Feature distribution shift (non-parametric) | p < 0.01 |
| ADWIN | Concept drift in streaming context | Adaptive window |
| JS Divergence | Symmetric KL divergence | JSD > threshold |
| Performance drift | Outcome gap champion vs expected | Gap > tolerance |
| Schema drift | Feature type/null changes | Any mismatch |
| Per-segment | Stratified drift per segment | Per-segment PSI |

### Retraining Pipeline

```
Drift detected → Retraining flow (Metaflow @kubernetes) → OPE gate
  → Shadow scoring → Canary ramp → Champion swap
```

Hot-challenger: Retraining runs on a 2h cadence regardless of drift.
Drift-fire is a metadata alias swap (`latest_candidate` → `challenger`),
eliminating the drift-to-deploy gap.

## 12. Customer Features Roadmap <a id='12-customer-features'></a>

### Current State: Behavioral Features Only

The model uses 21 behavioral features computed from the transaction stream.
These capture *what the customer is doing right now*.

### Missing: Customer-Level (Static) Features

In production credit decisioning, the model would also consume:

| Feature | Source | Update Frequency | Expected Lift |
|---------|--------|------------------|---------------|
| Credit bureau score | Equifax/Experian/TU | Daily pull | High — strongest risk predictor |
| Income bracket | CRM / payroll verification | Quarterly | Medium — capacity signal |
| Account tenure (months) | Core banking | Static | Medium — cold-start mitigation |
| Number of products | CRM | Monthly | Low-medium — cross-sell signal |
| Employment status | CRM / payroll | Quarterly | Medium — stability signal |
| Previous delinquency count | Core banking | Monthly | High — behavioral history |
| Payment-to-income ratio | Derived | Monthly | High — capacity utilization |

### How to Add Customer Features

1. **Create a new SageMaker Feature Group** (`customer_profile_features`)
2. **Write a batch Spark job** that reads from CRM/profile DB → writes to Feature Group
3. **Update `feature_lookup.py`** to fetch from both Feature Groups
4. **Update `feature_schema.json`** with new columns
5. **Retrain** with expanded feature set → deploy as challenger → shadow → promote

The architecture supports this seamlessly — SageMaker Feature Store handles
multi-group lookups, and the model pipeline accepts arbitrary feature columns.

In [ ]:
# Simulate what customer features would look like
np.random.seed(MASTER_SEED)

n_customers = 1000
customer_profiles = pd.DataFrame(
    {
        'customer_id': [f'cust-{str(i).zfill(6)}' for i in range(n_customers)],
        'credit_score': np.random.normal(700, 60, n_customers)
        .clip(300, 850)
        .astype(int),
        'income_bracket': np.random.choice(
            ['<30k', '30-60k', '60-100k', '100k+'],
            n_customers,
            p=[0.15, 0.35, 0.35, 0.15],
        ),
        'account_tenure_months': np.random.exponential(36, n_customers)
        .clip(1, 240)
        .astype(int),
        'n_products': np.random.poisson(2, n_customers).clip(1, 8),
        'prev_delinquency_count': np.random.poisson(0.3, n_customers).clip(0, 5),
        'employment_status': np.random.choice(
            ['employed', 'self_employed', 'retired', 'other'],
            n_customers,
            p=[0.65, 0.20, 0.10, 0.05],
        ),
    }
)

print('=== Simulated Customer Profile Features ===')
display(customer_profiles.describe())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(
    customer_profiles['credit_score'],
    bins=30,
    color='steelblue',
    alpha=0.7,
    edgecolor='black',
)
axes[0].set_title('Credit Score Distribution')
axes[0].set_xlabel('Credit Score')

customer_profiles['income_bracket'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='coral', alpha=0.7
)
axes[1].set_title('Income Bracket Distribution')
axes[1].tick_params(axis='x', rotation=45)

axes[2].hist(
    customer_profiles['account_tenure_months'],
    bins=30,
    color='green',
    alpha=0.7,
    edgecolor='black',
)
axes[2].set_title('Account Tenure (months)')
axes[2].set_xlabel('Months')

plt.suptitle(
    'Customer Profile Features (simulated — would come from CRM in production)',
    fontsize=13,
)
plt.tight_layout()
plt.show()

print('\nThese features would be stored in SageMaker Feature Store as a separate')
print('Feature Group and joined with the 21 behavioral features at inference time.')
print('Expected model lift: significant — credit score alone typically adds 5-15%')
print('AUC improvement in credit risk models.')

## Summary

### What We Built
- Per-segment T-learner uplift model with 3 heads (accept, spend, default)
- Optuna HPO with TPE sampler, fully logged to MLflow (all trials + best params)
- 5 validation mechanisms (synthetic ground-truth, holdout regression, OPE, shadow scoring, A/B documented)
- ONNX export with numerical equivalence verification
- Champion-challenger deployment with canary ramp
- 7-detector drift monitoring

### Key Results
| Metric | Value |
|--------|-------|
| Champion model | Logistic T-learner (τ = 0.43) |
| Simulated profit/decision | $302.05 (3.4× random) |
| /decide p50 latency | 7.11 ms |
| Total tests | 127 passing |
| ONNX equivalence | All segments < 1e-3 max diff |

### What Would Be Different in Production
1. **Customer features** from CRM/bureau — biggest expected lift
2. **More training data** — production scale (~1M+ customers, not 1K)
3. **Randomized holdout** (5%) for unbiased A/B evaluation
4. **Model monitoring** with real outcomes (not synthetic)
5. **Regulatory review** (SR 11-7 MRM, ECOA fair lending analysis)